# 02 — Weather Analysis

Analysis of weather forecast data for Eagle Ridge Flying Site.

**Goals:**
- Inspect Open-Meteo forecast data for the Eagle Ridge area
- Validate the thermal index formula against observed climb data
- Explore seasonal patterns in wind, temperature, and cloud cover
- Identify the most predictive weather features for thermal activity

**Advisory**: For research purposes only.

In [ ]:
import sys
sys.path.insert(0, '../backend')

import asyncio
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import datetime, timezone

from config.settings import get_settings
from data_ingestion.weather.mock_provider import MockWeatherProvider
from data_ingestion.weather.open_meteo import OpenMeteoProvider
from agents.weather_agent import WeatherAgent

settings = get_settings()
SITE_LAT = 35.492
SITE_LON = -118.187

print('Weather analysis notebook ready.')
print(f'Weather provider: {settings.weather_provider}')

## 1. Fetch Forecast Data

In [ ]:
# Use mock provider for offline development; switch to OpenMeteoProvider for real data
provider = MockWeatherProvider()

forecast = asyncio.run(provider.get_forecast(lat=SITE_LAT, lon=SITE_LON, days=3))

# Convert to DataFrame
rows = []
for h in forecast.hourly:
    rows.append({
        'time': pd.to_datetime(h.time),
        'temp_c': h.temp_c,
        'dewpoint_c': h.dewpoint_c,
        'spread': h.temp_c - h.dewpoint_c,
        'wind_speed_kmh': h.wind_speed_kmh,
        'wind_dir_deg': h.wind_dir_deg,
        'cloud_cover_pct': h.cloud_cover_pct,
        'humidity_pct': h.humidity_pct,
        'hour_utc': pd.to_datetime(h.time).hour,
    })

df = pd.DataFrame(rows)
df = df.set_index('time')

print(f'Loaded {len(df)} hourly records over {df.index.date[-1] - df.index.date[0]} days')
df.head(10)

## 2. Diurnal Pattern

In [ ]:
diurnal = df.groupby('hour_utc').agg({
    'temp_c': 'mean',
    'dewpoint_c': 'mean',
    'spread': 'mean',
    'wind_speed_kmh': 'mean',
    'cloud_cover_pct': 'mean',
})

fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)

# Temperature and dewpoint
ax = axes[0]
ax.plot(diurnal.index, diurnal['temp_c'], color='#ef4444', label='Temperature')
ax.plot(diurnal.index, diurnal['dewpoint_c'], color='#2563eb', label='Dewpoint')
ax.fill_between(diurnal.index, diurnal['dewpoint_c'], diurnal['temp_c'], alpha=0.15, color='#f97316', label='Spread')
ax.set_ylabel('Temperature (°C)')
ax.legend()
ax.set_title('Diurnal Temperature Pattern — Eagle Ridge (mock data)')
ax.axvspan(10, 15, alpha=0.08, color='#f97316', label='Peak flying window')

# Wind speed
ax = axes[1]
ax.plot(diurnal.index, diurnal['wind_speed_kmh'], color='#10b981')
ax.axhline(12, color='#94a3b8', linestyle='--', linewidth=0.8, label='Min optimal (12 km/h)')
ax.axhline(28, color='#f59e0b', linestyle='--', linewidth=0.8, label='Max optimal (28 km/h)')
ax.set_ylabel('Wind speed (km/h)')
ax.legend(fontsize=8)
ax.set_title('Diurnal Wind Pattern')

# Cloud cover
ax = axes[2]
ax.plot(diurnal.index, diurnal['cloud_cover_pct'], color='#6366f1')
ax.axhline(60, color='#ef4444', linestyle='--', linewidth=0.8, label='Overdevelopment threshold (60%)')
ax.set_xlabel('Hour UTC')
ax.set_ylabel('Cloud cover (%)')
ax.legend(fontsize=8)
ax.set_title('Diurnal Cloud Pattern')

for ax in axes:
    ax.set_xticks(range(0, 24, 2))
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Thermal Index Distribution

In [ ]:
agent = WeatherAgent(site_id='eagle_ridge')

thermal_indices = []
for h in forecast.hourly:
    score = agent.score_hour(h)
    thermal_indices.append({'hour_utc': pd.to_datetime(h.time).hour, 'thermal_index': score})

ti_df = pd.DataFrame(thermal_indices)
ti_diurnal = ti_df.groupby('hour_utc')['thermal_index'].mean()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Diurnal thermal index
colors = ['#10b981' if v >= 0.55 else '#f59e0b' if v >= 0.35 else '#ef4444'
          for v in ti_diurnal.values]
ax1.bar(ti_diurnal.index, ti_diurnal.values, color=colors, alpha=0.85)
ax1.axhline(0.55, color='#10b981', linestyle='--', linewidth=1, label='Green threshold (0.55)')
ax1.axhline(0.35, color='#f59e0b', linestyle='--', linewidth=1, label='Amber threshold (0.35)')
ax1.set_xlabel('Hour UTC')
ax1.set_ylabel('Thermal Index')
ax1.set_title('Thermal Index by Hour (avg across forecast)')
ax1.legend(fontsize=8)
ax1.set_xticks(range(0, 24, 2))

# Distribution
all_ti = ti_df['thermal_index'].values
ax2.hist(all_ti, bins=30, color='#2563eb', alpha=0.7, edgecolor='white')
ax2.axvline(0.55, color='#10b981', linestyle='--', label='Green ≥ 0.55')
ax2.axvline(0.35, color='#f59e0b', linestyle='--', label='Amber ≥ 0.35')
ax2.set_xlabel('Thermal Index')
ax2.set_ylabel('Count (hours)')
ax2.set_title('Thermal Index Distribution')
ax2.legend(fontsize=8)

plt.suptitle('WeatherAgent Thermal Index — Eagle Ridge', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Hours with thermal index ≥ 0.55: {(all_ti >= 0.55).sum()} / {len(all_ti)}')
print(f'Hours with thermal index ≥ 0.35: {(all_ti >= 0.35).sum()} / {len(all_ti)}')
print(f'Peak thermal index: {all_ti.max():.3f} at hour {ti_df.loc[ti_df["thermal_index"].idxmax(), "hour_utc"]:02d}:00 UTC')

## 4. Feature Correlation Matrix

In [ ]:
# Add thermal index to the main DataFrame
df_day = df[(df['hour_utc'] >= 6) & (df['hour_utc'] <= 20)].copy()
ti_series = [agent.score_hour(h) for h in forecast.hourly
             if 6 <= pd.to_datetime(h.time).hour <= 20]
df_day = df_day.iloc[:len(ti_series)].copy()
df_day['thermal_index'] = ti_series

corr = df_day[['temp_c', 'spread', 'wind_speed_kmh', 'cloud_cover_pct', 'humidity_pct', 'thermal_index']].corr()

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(corr, cmap='RdBu_r', vmin=-1, vmax=1)
plt.colorbar(im, ax=ax)

labels = corr.columns.tolist()
ax.set_xticks(range(len(labels)))
ax.set_yticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=45, ha='right')
ax.set_yticklabels(labels)

for i in range(len(labels)):
    for j in range(len(labels)):
        ax.text(j, i, f'{corr.iloc[i, j]:.2f}', ha='center', va='center',
                fontsize=9, color='white' if abs(corr.iloc[i, j]) > 0.5 else 'black')

ax.set_title('Weather Feature Correlation Matrix (daytime hours)')
plt.tight_layout()
plt.show()

## 5. Summary

Key findings:
- (Fill in after running with real data)
- Peak thermal index hour
- Most correlated features with thermal index
- Diurnal patterns matching local pilot knowledge

Next: `03_baseline_thermal_model.ipynb`